# Week11 예습과제 — 6장 임베딩 (Word2Vec ~ 순환 신경망)
『자연어처리와 컴퓨터비전 심층학습』 6장 코드 필사 (예제 6.3 ~ 6.30)

- 런타임: **T4 GPU** 권장

### 🔹 0. 환경 설정
코랩에는 Korpora·konlpy·gensim이 없어서 먼저 설치한다.
설치 후 **"세션 다시 시작"** 안내가 뜨면 런타임을 재시작하고 다음 셀부터 실행한다.

In [3]:
!pip install Korpora konlpy gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.4/19.4 MB 39.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 125.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [gensim]


### 🔹 예제 6.3 기본 Skip-gram 클래스
- `embedding` = W(V×E): 단어 인덱스로 임베딩을 꺼내는 룩업
- `linear` = W′(E×V): 임베딩을 단어별 점수로 바꿈

In [1]:
from torch import nn

class VanillaSkipgram(nn.Module):
    def __init__(self, vocab_size, embedding_dim):
        super().__init__()
        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embedding_dim
        )
        self.linear = nn.Linear(
            in_features=embedding_dim,
            out_features=vocab_size
        )

    def forward(self, input_ids):
        embeddings = self.embedding(input_ids)  # 단어 인덱스 → 임베딩 벡터 (W)
        output = self.linear(embeddings)        # 임베딩 → 단어별 점수 (W′)
        return output

### 🔹 예제 6.4 영화 리뷰 데이터세트 전처리
네이버 영화 리뷰(nsmc) 중 크기가 작은 테스트셋(5만 개)을 Okt로 형태소 분석한다.
형태소 분석에 **몇 분** 걸린다.

In [4]:
import pandas as pd
from Korpora import Korpora
from konlpy.tag import Okt

corpus = Korpora.load("nsmc")
corpus = pd.DataFrame(corpus.test)

tokenizer = Okt()
tokens = [tokenizer.morphs(review) for review in corpus.text]
print(tokens[:3])


    Korpora 는 다른 분들이 연구 목적으로 공유해주신 말뭉치들을
    손쉽게 다운로드, 사용할 수 있는 기능만을 제공합니다.

    말뭉치들을 공유해 주신 분들에게 감사드리며, 각 말뭉치 별 설명과 라이센스를 공유 드립니다.
    해당 말뭉치에 대해 자세히 알고 싶으신 분은 아래의 description 을 참고,
    해당 말뭉치를 연구/상용의 목적으로 이용하실 때에는 아래의 라이센스를 참고해 주시기 바랍니다.

    # Description
    Author : e9t@github
    Repository : https://github.com/e9t/nsmc
    References : www.lucypark.kr/docs/2015-pyconkr/#39

    Naver sentiment movie corpus v1.0
    This is a movie review dataset in the Korean language.
    Reviews were scraped from Naver Movies.

    The dataset construction is based on the method noted in
    [Large movie review dataset][^1] from Maas et al., 2011.

    [^1]: http://ai.stanford.edu/~amaas/data/sentiment/

    # License
    CC0 1.0 Universal (CC0 1.0) Public Domain Dedication
    Details in https://creativecommons.org/publicdomain/zero/1.0/



[nsmc] download ratings_train.txt: 14.6MB [00:00, 51.6MB/s]                            
[nsmc] download ratings_test.txt: 4.90MB [00:00, 33.7MB/s]                            


[['굳', 'ㅋ'], ['GDNTOPCLASSINTHECLUB'], ['뭐', '야', '이', '평점', '들', '은', '....', '나쁘진', '않지만', '10', '점', '짜', '리', '는', '더', '더욱', '아니잖아']]


### 🔹 예제 6.5 단어 사전 구축
가장 많이 나온 5,000개 + 특수 토큰 `<unk>` 1개 = 5,001개

In [7]:
from collections import Counter

def build_vocab(corpus, n_vocab, special_tokens):
    counter = Counter()
    for tokens in corpus:
        counter.update(tokens)
    vocab = special_tokens
    for token, count in counter.most_common(n_vocab):
        vocab.append(token)
    return vocab


vocab = build_vocab(corpus=tokens, n_vocab=5000, special_tokens=["<unk>"])
token_to_id = {token: idx for idx, token in enumerate(vocab)}
id_to_token = {idx: token for idx, token in enumerate(vocab)}

print(vocab[:10])
print(len(vocab))

['<unk>', '.', '이', '영화', '의', '..', '가', '에', '...', '을']
5001


### 🔹 예제 6.6 Skip-gram의 단어 쌍 추출
중심 단어의 앞뒤 2개(window_size=2)를 주변 단어로 → [중심 단어, 주변 단어] 쌍

In [5]:
def get_word_pairs(tokens, window_size):
    pairs = []
    for sentence in tokens:
        sentence_length = len(sentence)
        for idx, center_word in enumerate(sentence):
            window_start = max(0, idx - window_size)                    # 문장 앞을 넘어가지 않게
            window_end = min(sentence_length, idx + window_size + 1)    # 문장 끝을 넘어가지 않게
            center_word = sentence[idx]
            context_words = sentence[window_start:idx] + sentence[idx+1:window_end]
            for context_word in context_words:
                pairs.append([center_word, context_word])
    return pairs


word_pairs = get_word_pairs(tokens, window_size=2)
print(word_pairs[:5])

[['굳', 'ㅋ'], ['ㅋ', '굳'], ['뭐', '야'], ['뭐', '이'], ['야', '뭐']]


### 🔹 예제 6.7 인덱스 쌍 변환
임베딩 층은 단어 인덱스를 받으므로 단어 쌍 → 인덱스 쌍. 사전에 없으면 `<unk>` 인덱스.

In [8]:
def get_index_pairs(word_pairs, token_to_id):
    pairs = []
    unk_index = token_to_id["<unk>"]
    for word_pair in word_pairs:
        center_word, context_word = word_pair
        center_index = token_to_id.get(center_word, unk_index)    # 없으면 <unk>
        context_index = token_to_id.get(context_word, unk_index)
        pairs.append([center_index, context_index])
    return pairs


index_pairs = get_index_pairs(word_pairs, token_to_id)
print(index_pairs[:5])

[[595, 100], [100, 595], [77, 176], [77, 2], [176, 77]]


### 🔹 예제 6.8 데이터로더 적용
[N, 2] 텐서를 중심 단어(0번 열)와 주변 단어(1번 열)로 나눠 데이터세트로 만든다.

In [9]:
import torch
from torch.utils.data import TensorDataset, DataLoader

index_pairs = torch.tensor(index_pairs)
center_indexs = index_pairs[:, 0]
contenxt_indexs = index_pairs[:, 1]   # 교재 변수명 그대로 (contenxt 오타)

dataset = TensorDataset(center_indexs, contenxt_indexs)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

### 🔹 예제 6.9 Skip-gram 모델 준비 작업
단어 사전 크기만큼 클래스가 있는 분류 문제 → 교차 엔트로피 (내부에서 소프트맥스 수행)

In [10]:
from torch import optim

device = "cuda" if torch.cuda.is_available() else "cpu"
word2vec = VanillaSkipgram(vocab_size=len(token_to_id), embedding_dim=128).to(device)
criterion = nn.CrossEntropyLoss().to(device)
optimizer = optim.SGD(word2vec.parameters(), lr=0.1)

### 🔹 예제 6.10 모델 학습
- 배치 수가 많아 **에폭당 몇 분**, 10에폭이면 수십 분 걸린다. 시간이 부족하면 `range(10)`의 숫자를 줄여도 된다.

In [11]:
for epoch in range(10):
    cost = 0.0
    for input_ids, target_ids in dataloader:
        input_ids = input_ids.to(device)
        target_ids = target_ids.to(device)

        logits = word2vec(input_ids)
        loss = criterion(logits, target_ids)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        cost += loss.item()   # [수정] 교재: cost += loss

    cost = cost / len(dataloader)
    print(f"Epoch : {epoch+1:4d}, Cost : {cost:.3f}")

Epoch :    1, Cost : 6.196
Epoch :    2, Cost : 5.982
Epoch :    3, Cost : 5.932
Epoch :    4, Cost : 5.902
Epoch :    5, Cost : 5.880
Epoch :    6, Cost : 5.862
Epoch :    7, Cost : 5.847
Epoch :    8, Cost : 5.834
Epoch :    9, Cost : 5.823
Epoch :   10, Cost : 5.812


### 🔹 예제 6.11 임베딩 값 추출
학습이 끝난 W(V×E) = 임베딩 층의 가중치를 꺼내 단어별 임베딩으로 쓴다.

In [12]:
token_to_embedding = dict()
embedding_matrix = word2vec.embedding.weight.detach().cpu().numpy()   # [5001, 128]

for word, embedding in zip(vocab, embedding_matrix):
    token_to_embedding[word] = embedding

index = 30
token = vocab[30]
token_embedding = token_to_embedding[token]
print(token)
print(token_embedding)

연기
[-1.521324   -1.0077432   0.85779095  0.6040343   2.2331424  -0.13498035
  0.8582767   0.43663543 -0.27492654  1.1679498  -0.7746241   1.7761481
  0.3143711   0.19209276 -0.5949139   0.64802366 -1.0694503   0.7305204
 -0.6024521   0.82705516  0.5231024   0.39116815  0.5849611  -0.2349108
  0.43785572  0.3276464  -0.638045    0.18362361 -0.35008895  0.00816453
 -1.7098114   1.1189808  -0.19696152  1.8124436   0.5985938   1.0467576
 -1.1213664  -0.2086813  -0.67915076  0.7600985  -0.39228657  0.133932
  0.60077155  0.48032594  1.7040019   0.30419293  0.5873318  -1.149216
  1.8129046   0.6194227   0.34919086  1.2378142  -0.18050575 -0.17769116
  0.76676315 -1.6174412   0.31156832 -0.72436875 -0.29608086  0.16456746
  1.3584236  -2.1050391  -0.2294515  -0.2097543   0.19930059  0.6996807
 -1.3806586  -0.7002395   0.7004672   0.9046365   1.2833298   1.1841184
  1.2868004  -0.9427408  -0.28525048 -0.4094888  -0.04056302  0.5211613
  0.60768974 -0.819168   -0.6407198  -1.2344575  -0.1943658

### 🔹 예제 6.12 단어 임베딩 유사도 계산
코사인 유사도 = 내적 ÷ (두 벡터 크기의 곱). 1위는 자기 자신이라 `[1 : n+1]`로 두 번째부터 반환.

In [13]:
import numpy as np
from numpy.linalg import norm

def cosine_similarity(a, b):
    cosine = np.dot(b, a) / (norm(b, axis=1) * norm(a))   # a: 단어 1개, b: 사전 전체 → 한 번에 계산
    return cosine

def top_n_index(cosine_matrix, n):
    closest_indexes = cosine_matrix.argsort()[::-1]   # 유사도 내림차순
    top_n = closest_indexes[1 : n + 1]                # 0번(자기 자신) 제외
    return top_n


cosine_matrix = cosine_similarity(token_embedding, embedding_matrix)
top_n = top_n_index(cosine_matrix, n=5)

print(f"{token}와 가장 유사한 5 개 단어")
for index in top_n:
    print(f"{id_to_token[index]} - 유사도 : {cosine_matrix[index]:.4f}")

연기와 가장 유사한 5 개 단어
전도연 - 유사도 : 0.3145
떼 - 유사도 : 0.3015
대사 - 유사도 : 0.3014
,. - 유사도 : 0.3002
어색함 - 유사도 : 0.2974


## 모델 실습: Gensim
### 🔹 Word2Vec 클래스 (참고용)
기본 매개변수 구조. 실행용이 아니라 텍스트 셀에 기록한다.

### 🔹 예제 6.13 Word2Vec 모델 학습
Skip-gram(sg=1), 128차원, 윈도 5, 최소 빈도 무시, 최대 사전 10,000, 에폭 3
- `[수정]` 교재는 저장 줄이 주석이지만, **예제 6.28에서 이 모델을 불러오므로** 저장해 둔다.

In [14]:
import os
from gensim.models import Word2Vec

word2vec = Word2Vec(
    sentences=tokens,
    vector_size=128,
    window=5,
    min_count=1,
    sg=1,
    epochs=3,
    max_final_vocab=10000
)

os.makedirs("../models", exist_ok=True)       # [추가] 저장 폴더 만들기
word2vec.save("../models/word2vec.model")     # [수정] 주석 해제 (예제 6.28에서 사용)
# word2vec = Word2Vec.load("../models/word2vec.model")

### 🔹 예제 6.14 임베딩 추출 및 유사도 계산
`wv` = 학습된 단어 벡터 모음 (Word2VecKeyedVectors)

In [15]:
word = "연기"
print(word2vec.wv[word])
print(word2vec.wv.most_similar(word, topn=5))
print(word2vec.wv.similarity(w1=word, w2="연기력"))

[ 0.06994002 -0.36459684  0.25062248  0.5023641  -0.0369399   0.04221897
  0.15610486  0.00754895 -0.5156513   0.26118848 -0.12612197 -0.2998824
 -0.30684942  0.00836504 -0.3412122  -0.1715759  -0.49252903  0.398227
 -0.3406544   0.51253915  0.47566694  0.3992519  -0.19436143 -0.18657744
  0.03870013 -0.02444562 -0.49582958 -0.13399816  0.1001242  -0.12490641
 -0.31799734  0.31787232  0.2785136  -0.1516584  -0.03804904 -0.29728976
  0.2803325  -0.27312142 -0.04677587 -0.3989977  -0.04481955  0.29106838
 -0.12351398 -0.35310942 -0.19372986  0.16712064 -0.204198   -0.37282684
 -0.03521764  0.12749213  0.8950651   0.33459994  0.09666824  0.28014266
 -0.49384055 -0.41182292  0.12487549  0.00546788 -0.28604597  0.12325954
 -0.07056904 -0.11353409  0.02341218  0.08018047 -0.5341633   0.03686612
  0.01940605  0.34188098  0.08819754 -0.16082183 -0.18735535 -0.4239108
 -0.4048918  -0.00461674  0.08544261  0.02232487 -0.28313965 -0.5208153
 -0.08962684  0.5861603   0.01164551 -0.06945219  0.1287

## 2. fastText
### 🔹 FastText 클래스 (참고용)
Word2Vec 매개변수 + N-gram 범위(`min_n`, `max_n`). 실행용이 아니라 텍스트 셀에 기록한다.

### 🔹 예제 6.15 KorNLI 데이터세트 전처리
fastText는 단어 구조를 스스로 학습하므로 형태소 분석 없이 **띄어쓰기(`split`)**로 토큰화한다.
- 여기서 `tokens`가 KorNLI 토큰으로 바뀐다. (nsmc로 학습한 Word2Vec은 6.13에서 이미 저장해 둠)

In [16]:
from Korpora import Korpora

corpus = Korpora.load("kornli")
corpus_texts = corpus.get_all_texts() + corpus.get_all_pairs()
tokens = [sentence.split() for sentence in corpus_texts]

print(tokens[:3])


    Korpora 는 다른 분들이 연구 목적으로 공유해주신 말뭉치들을
    손쉽게 다운로드, 사용할 수 있는 기능만을 제공합니다.

    말뭉치들을 공유해 주신 분들에게 감사드리며, 각 말뭉치 별 설명과 라이센스를 공유 드립니다.
    해당 말뭉치에 대해 자세히 알고 싶으신 분은 아래의 description 을 참고,
    해당 말뭉치를 연구/상용의 목적으로 이용하실 때에는 아래의 라이센스를 참고해 주시기 바랍니다.

    # Description
    Author : KakaoBrain
    Repository : https://github.com/kakaobrain/KorNLUDatasets
    References :
        - Ham, J., Choe, Y. J., Park, K., Choi, I., & Soh, H. (2020). KorNLI and KorSTS: New Benchmark
           Datasets for Korean Natural Language Understanding. arXiv preprint arXiv:2004.03289.
           (https://arxiv.org/abs/2004.03289)

    This is the dataset repository for our paper
    "KorNLI and KorSTS: New Benchmark Datasets for Korean Natural Language Understanding."
    (https://arxiv.org/abs/2004.03289)
    We introduce KorNLI and KorSTS, which are NLI and STS datasets in Korean.

    # License
    Creative Commons Attribution-ShareAlike license (CC BY-SA 4.0)
    Details in https://creativecommons.org/licenses

[kornli] download multinli.train.ko.tsv: 83.6MB [00:00, 97.1MB/s]                           
[kornli] download snli_1.0_train.ko.tsv: 78.5MB [00:00, 143MB/s]                            
[kornli] download xnli.dev.ko.tsv: 516kB [00:00, 9.38MB/s]
[kornli] download xnli.test.ko.tsv: 1.04MB [00:00, 17.0MB/s]


[['개념적으로', '크림', '스키밍은', '제품과', '지리라는', '두', '가지', '기본', '차원을', '가지고', '있다.'], ['시즌', '중에', '알고', '있는', '거', '알아?', '네', '레벨에서', '다음', '레벨로', '잃어버리는', '거야', '브레이브스가', '모팀을', '떠올리기로', '결정하면', '브레이브스가', '트리플', 'A에서', '한', '남자를', '떠올리기로', '결정하면', '더블', 'A가', '그를', '대신하러', '올라가고', 'A', '한', '명이', '그를', '대신하러', '올라간다.'], ['우리', '번호', '중', '하나가', '당신의', '지시를', '세밀하게', '수행할', '것이다.']]


### 🔹 예제 6.16 fastText 모델 실습
128차원, 윈도 5, 5번 이상 나온 단어만, N-gram 범위 [2, 6], Skip-gram, 에폭 3

In [17]:
from gensim.models import FastText

fastText = FastText(
    sentences=tokens,
    vector_size=128,
    window=5,
    min_count=5,
    sg=1,
    epochs=3,
    min_n=2,
    max_n=6
)

# fastText.save("../models/fastText.model")
# fastText = FastText.load("../models/fastText.model")

### 🔹 예제 6.17 fastText OOV 처리
'사랑해요'는 사전에 없지만(False) 하위 단어(사랑, 랑해, 해요 …)를 합쳐 임베딩을 만든다.

## 3. 순환 신경망
### 🔹 순환 신경망 클래스 (참고용)
`input_size`, `hidden_size` 값이 정해져 있지 않아 텍스트 셀에 기록한다.


### 🔹 예제 6.18 양방향 다층 신경망
- 입력 [4, 6, 128], 초기 은닉 상태 [3층 × 2방향, 4, 256]
- 출력 [4, 6, 2×256 = 512], 최종 은닉 상태 [6, 4, 256]

In [18]:
import torch
from torch import nn

input_size = 128
ouput_size = 256
num_layers = 3
bidirectional = True

model = nn.RNN(
    input_size=input_size,
    hidden_size=ouput_size,
    num_layers=num_layers,
    nonlinearity="tanh",
    batch_first=True,
    bidirectional=bidirectional,
)

batch_size = 4
sequence_len = 6

inputs = torch.randn(batch_size, sequence_len, input_size)
h_0 = torch.rand(num_layers * (int(bidirectional) + 1), batch_size, ouput_size)   # [3*2, 4, 256]

outputs, hidden = model(inputs, h_0)
print(outputs.shape)
print(hidden.shape)

torch.Size([4, 6, 512])
torch.Size([6, 4, 256])


### 🔹 예제 6.19 양방향 다층 장단기 메모리
- 입력은 `(h_0, c_0)` 튜플, 반환도 `outputs, (h_n, c_n)`
- proj_size=64 → h는 64, c는 은닉 크기 256 그대로
- 출력 [4, 6, 2×64 = 128], h_n [6, 4, 64], c_n [6, 4, 256]

In [19]:
import torch
from torch import nn

input_size = 128
ouput_size = 256
num_layers = 3
bidirectional = True
proj_size = 64

model = nn.LSTM(
    input_size=input_size,
    hidden_size=ouput_size,
    num_layers=num_layers,
    batch_first=True,
    bidirectional=bidirectional,
    proj_size=proj_size,
)

batch_size = 4
sequence_len = 6

inputs = torch.randn(batch_size, sequence_len, input_size)
h_0 = torch.rand(
    num_layers * (int(bidirectional) + 1),
    batch_size,
    proj_size if proj_size > 0 else ouput_size,   # 투사를 쓰면 h는 투사 크기
)
c_0 = torch.rand(num_layers * (int(bidirectional) + 1), batch_size, ouput_size)   # c는 항상 은닉 크기

outputs, (h_n, c_n) = model(inputs, (h_0, c_0))

print(outputs.shape)
print(h_n.shape)
print(c_n.shape)

torch.Size([4, 6, 128])
torch.Size([6, 4, 64])
torch.Size([6, 4, 256])


/usr/local/lib/python3.13/dist-packages/torch/nn/modules/rnn.py:1164: UserWarning: LSTM with projections is not supported with oneDNN. Using default implementation. (Triggered internally at /__w/pytorch/pytorch/aten/src/ATen/native/RNN.cpp:1512.)
  result = _VF.lstm(


## 모델 실습: 문장 긍/부정 분류
### 🔹 예제 6.20 문장 분류 모델
임베딩 → RNN/LSTM → **마지막 시점 출력** `[:, -1, :]` → 드롭아웃 → 분류기(출력 1개)
- 양방향이면 출력이 2배라서 분류기 입력이 `hidden_dim * 2`

In [20]:
from torch import nn

class SentenceClassifier(nn.Module):
    def __init__(
        self,
        n_vocab,
        hidden_dim,
        embedding_dim,
        n_layers,
        dropout=0.5,
        bidirectional=True,
        model_type="lstm"
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            num_embeddings=n_vocab,
            embedding_dim=embedding_dim,
            padding_idx=0
        )
        if model_type == "rnn":
            self.model = nn.RNN(
                input_size=embedding_dim,
                hidden_size=hidden_dim,
                num_layers=n_layers,
                bidirectional=bidirectional,
                dropout=dropout,
                batch_first=True,
            )
        elif model_type == "lstm":
            self.model = nn.LSTM(
                input_size=embedding_dim,
                hidden_size=hidden_dim,
                num_layers=n_layers,
                bidirectional=bidirectional,
                dropout=dropout,
                batch_first=True,
            )

        if bidirectional:
            self.classifier = nn.Linear(hidden_dim * 2, 1)
        else:
            self.classifier = nn.Linear(hidden_dim, 1)
        self.dropout = nn.Dropout(dropout)

    def forward(self, inputs):
        embeddings = self.embedding(inputs)
        output, _ = self.model(embeddings)
        last_output = output[:, -1, :]          # 마지막 시점의 출력만 사용 (다대일)
        last_output = self.dropout(last_output)
        logits = self.classifier(last_output)
        return logits

### 🔹 예제 6.21 데이터세트 불러오기
nsmc 테스트셋을 학습 90% / 테스트 10%로 나눈다.

In [21]:
import pandas as pd
from Korpora import Korpora

corpus = Korpora.load("nsmc")
corpus_df = pd.DataFrame(corpus.test)

train = corpus_df.sample(frac=0.9, random_state=42)
test = corpus_df.drop(train.index)

print(train.head(5).to_markdown())
print("Training Data Size :", len(train))
print("Testing Data Size :", len(test))


    Korpora 는 다른 분들이 연구 목적으로 공유해주신 말뭉치들을
    손쉽게 다운로드, 사용할 수 있는 기능만을 제공합니다.

    말뭉치들을 공유해 주신 분들에게 감사드리며, 각 말뭉치 별 설명과 라이센스를 공유 드립니다.
    해당 말뭉치에 대해 자세히 알고 싶으신 분은 아래의 description 을 참고,
    해당 말뭉치를 연구/상용의 목적으로 이용하실 때에는 아래의 라이센스를 참고해 주시기 바랍니다.

    # Description
    Author : e9t@github
    Repository : https://github.com/e9t/nsmc
    References : www.lucypark.kr/docs/2015-pyconkr/#39

    Naver sentiment movie corpus v1.0
    This is a movie review dataset in the Korean language.
    Reviews were scraped from Naver Movies.

    The dataset construction is based on the method noted in
    [Large movie review dataset][^1] from Maas et al., 2011.

    [^1]: http://ai.stanford.edu/~amaas/data/sentiment/

    # License
    CC0 1.0 Universal (CC0 1.0) Public Domain Dedication
    Details in https://creativecommons.org/publicdomain/zero/1.0/

[Korpora] Corpus `nsmc` is already installed at /root/Korpora/nsmc/ratings_train.txt
[Korpora] Corpus `nsmc` is already installed at /root/Korpora/nsmc/ra

### 🔹 예제 6.22 데이터 토큰화 및 단어 사전 구축
문장 길이를 맞추기 위해 `<pad>`를 추가 → `<pad>`=0, `<unk>`=1, 사전 크기 5,002

In [22]:
from konlpy.tag import Okt
from collections import Counter

def build_vocab(corpus, n_vocab, special_tokens):
    counter = Counter()
    for tokens in corpus:
        counter.update(tokens)
    vocab = special_tokens
    for token, count in counter.most_common(n_vocab):
        vocab.append(token)
    return vocab

tokenizer = Okt()
train_tokens = [tokenizer.morphs(review) for review in train.text]
test_tokens = [tokenizer.morphs(review) for review in test.text]

vocab = build_vocab(corpus=train_tokens, n_vocab=5000, special_tokens=["<pad>", "<unk>"])
token_to_id = {token: idx for idx, token in enumerate(vocab)}
id_to_token = {idx: token for idx, token in enumerate(vocab)}

print(vocab[:10])
print(len(vocab))

['<pad>', '<unk>', '.', '이', '영화', '의', '..', '가', '에', '...']
5002


### 🔹 예제 6.23 정수 인코딩 및 패딩
- 사전에 없는 단어 → 1(`<unk>`)
- 32보다 길면 자르고, 짧으면 뒤에 0(`<pad>`)을 붙여 길이 32로 맞춘다.

In [23]:
import numpy as np

def pad_sequences(sequences, max_length, pad_value):
    result = list()
    for sequence in sequences:
        sequence = sequence[:max_length]                          # 길면 자르기
        pad_length = max_length - len(sequence)
        padded_sequence = sequence + [pad_value] * pad_length     # 짧으면 뒤에 <pad> 붙이기
        result.append(padded_sequence)
    return np.asarray(result)

unk_id = token_to_id["<unk>"]
train_ids = [
    [token_to_id.get(token, unk_id) for token in review] for review in train_tokens
]
test_ids = [
    [token_to_id.get(token, unk_id) for token in review] for review in test_tokens
]

max_length = 32
pad_id = token_to_id["<pad>"]
train_ids = pad_sequences(train_ids, max_length, pad_id)
test_ids = pad_sequences(test_ids, max_length, pad_id)

print(train_ids[0])
print(test_ids[0])

[ 223 1716   10 4036 2095  193  755    4    2 2330 1031  220   26   13
 4839    1    1    1    2    0    0    0    0    0    0    0    0    0
    0    0    0    0]
[3307    5 1997  456    8    1 1013 3906    5    1    1   13  223   51
    3    1 4684    6    0    0    0    0    0    0    0    0    0    0
    0    0    0    0]


### 🔹 예제 6.24 데이터로더 적용
라벨은 BCE 손실에 맞춰 `float32`로 변환

In [24]:
import torch
from torch.utils.data import TensorDataset, DataLoader

train_ids = torch.tensor(train_ids)
test_ids = torch.tensor(test_ids)

train_labels = torch.tensor(train.label.values, dtype=torch.float32)
test_labels = torch.tensor(test.label.values, dtype=torch.float32)

train_dataset = TensorDataset(train_ids, train_labels)
test_dataset = TensorDataset(test_ids, test_labels)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

### 🔹 예제 6.25 손실 함수와 최적화 함수 정의
- 은닉 64, 임베딩 128, 2층 (기본값: 양방향 LSTM)
- `BCEWithLogitsLoss` = BCELoss + Sigmoid / `RMSprop` = 지수 가중 이동 평균으로 학습률 조절

In [25]:
from torch import optim

n_vocab = len(token_to_id)
hidden_dim = 64
embedding_dim = 128
n_layers = 2

device = "cuda" if torch.cuda.is_available() else "cpu"
classifier = SentenceClassifier(
    n_vocab=n_vocab, hidden_dim=hidden_dim, embedding_dim=embedding_dim, n_layers=n_layers
).to(device)
criterion = nn.BCEWithLogitsLoss().to(device)
optimizer = optim.RMSprop(classifier.parameters(), lr=0.001)

### 🔹 예제 6.26 모델 학습 및 테스트 (1) 함수 정의
- `unsqueeze(1)`: 라벨 [16] → [16, 1]로 모델 출력 모양과 맞춤
- 테스트: sigmoid 값이 0.5보다 크면 긍정(1)으로 예측

In [26]:
def train(model, datasets, criterion, optimizer, device, interval):
    model.train()
    losses = list()

    for step, (input_ids, labels) in enumerate(datasets):
        input_ids = input_ids.to(device)
        labels = labels.to(device).unsqueeze(1)   # [16] → [16, 1]

        logits = model(input_ids)
        loss = criterion(logits, labels)
        losses.append(loss.item())

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if step % interval == 0:
            print(f"Train Loss {step} : {np.mean(losses)}")


def test(model, datasets, criterion, device):
    model.eval()
    losses = list()
    corrects = list()

    for step, (input_ids, labels) in enumerate(datasets):
        input_ids = input_ids.to(device)
        labels = labels.to(device).unsqueeze(1)

        logits = model(input_ids)
        loss = criterion(logits, labels)
        losses.append(loss.item())
        yhat = torch.sigmoid(logits)>.5           # 확률 0.5 초과 → 긍정
        corrects.extend(
            torch.eq(yhat, labels).cpu().tolist()
        )

    print(f"Val Loss : {np.mean(losses)}, Val Accuracy : {np.mean(corrects)}")

### 🔹 예제 6.26 모델 학습 및 테스트 (2) 학습 실행
교재 결과: Val Accuracy 0.7622 (1에폭) → 0.8236 (5에폭)

In [27]:
epochs = 5
interval = 500

for epoch in range(epochs):
    train(classifier, train_loader, criterion, optimizer, device, interval)
    test(classifier, test_loader, criterion, device)

Train Loss 0 : 0.6715505719184875
Train Loss 500 : 0.6934190864810449
Train Loss 1000 : 0.6853646206093597
Train Loss 1500 : 0.6768661587139831
Train Loss 2000 : 0.664898711493586
Train Loss 2500 : 0.649718939733715
Val Loss : 0.5553765297888186, Val Accuracy : 0.7356
Train Loss 0 : 0.42881470918655396
Train Loss 500 : 0.5359981168292002
Train Loss 1000 : 0.526564449711994
Train Loss 1500 : 0.5115762257838075
Train Loss 2000 : 0.5024114194093854
Train Loss 2500 : 0.4961694643848755
Val Loss : 0.46911085961154475, Val Accuracy : 0.794
Train Loss 0 : 0.1908183991909027
Train Loss 500 : 0.43944522339902714
Train Loss 1000 : 0.4313631598408763
Train Loss 1500 : 0.4382637024303026
Train Loss 2000 : 0.43774806130593685
Train Loss 2500 : 0.43218436893798123
Val Loss : 0.43095512607227116, Val Accuracy : 0.8054
Train Loss 0 : 0.31251847743988037
Train Loss 500 : 0.37374853533125685
Train Loss 1000 : 0.3710164939993983
Train Loss 1500 : 0.37351447013240907
Train Loss 2000 : 0.37229942557202284


### 🔹 예제 6.27 학습된 모델로부터 임베딩 추출

In [28]:
token_to_embedding = dict()
embedding_matrix = classifier.embedding.weight.detach().cpu().numpy()

for word, emb in zip(vocab, embedding_matrix):
    token_to_embedding[word] = emb

token = vocab[1000]
print(token, token_to_embedding[token])

보고싶다 [ 1.16914868e+00 -2.96833068e-01 -1.92567778e+00 -6.81923032e-01
  1.65280294e+00  9.70167890e-02 -7.86236286e-01 -1.32319689e+00
  8.22988749e-01  1.26786101e+00 -3.62914316e-02 -7.33981311e-01
  2.40999961e+00  2.46543026e+00 -5.26613891e-01 -5.85422516e-01
 -9.29188609e-01  1.34552085e+00  2.33653367e-01  5.67611873e-01
  6.61349595e-01 -2.23520136e+00  2.19554615e+00  5.02126515e-01
 -1.12748897e+00 -1.13890171e+00  1.45662332e+00 -7.24376738e-01
  9.30368423e-01  1.06518078e+00 -8.06188118e-03 -3.77276272e-01
 -9.52861384e-02  2.39427909e-02  1.99275345e-01  8.36904824e-01
 -2.01211154e-01 -1.20178509e+00  1.73885143e+00  3.06183934e-01
 -4.62606460e-01  1.87569275e-01 -5.00748456e-01  1.97610795e-03
 -6.04524761e-02  3.46277049e-03 -1.18028259e+00  4.24423814e-01
 -1.13301955e-01  8.20609033e-01 -5.65991580e-01 -6.70742691e-01
 -1.57865715e+00 -1.17664766e+00 -1.35521424e+00  1.02183986e+00
 -6.95395350e-01  1.17321968e+00 -1.56933820e+00  3.92389387e-01
  7.38201916e-01  6.

### 🔹 예제 6.28 사전 학습된 모델로 임베딩 계층 초기화
예제 6.13에서 저장한 Word2Vec(같은 nsmc 말뭉치, 128차원)을 불러와 [5002, 128] 배열을 채운다. `<pad>`, `<unk>`는 0 그대로 둔다.

In [29]:
from gensim.models import Word2Vec

word2vec = Word2Vec.load("../models/word2vec.model")
init_embeddings = np.zeros((n_vocab, embedding_dim))   # [5002, 128]

for index, token in id_to_token.items():
    if token not in ["<pad>", "<unk>"]:                # 특수 토큰은 초기화에서 제외
        init_embeddings[index] = word2vec.wv[token]

embedding_layer = nn.Embedding.from_pretrained(
    torch.tensor(init_embeddings, dtype=torch.float32)
)

### 🔹 예제 6.29 사전 학습된 임베딩 계층 적용

In [30]:
class SentenceClassifier(nn.Module):
    def __init__(
        self,
        n_vocab,
        hidden_dim,
        embedding_dim,
        n_layers,
        dropout=0.5,
        bidirectional=True,
        model_type="lstm",
        pretrained_embedding=None                     # ← 6.29 변경
    ):
        super().__init__()

        if pretrained_embedding is not None:          # ← 6.29 변경
            self.embedding = nn.Embedding.from_pretrained(
                torch.tensor(pretrained_embedding, dtype=torch.float32)
            )
        else:
            self.embedding = nn.Embedding(
                num_embeddings=n_vocab,
                embedding_dim=embedding_dim,
                padding_idx=0
            )

        if model_type == "rnn":
            self.model = nn.RNN(
                input_size=embedding_dim,
                hidden_size=hidden_dim,
                num_layers=n_layers,
                bidirectional=bidirectional,
                dropout=dropout,
                batch_first=True,
            )
        elif model_type == "lstm":
            self.model = nn.LSTM(
                input_size=embedding_dim,
                hidden_size=hidden_dim,
                num_layers=n_layers,
                bidirectional=bidirectional,
                dropout=dropout,
                batch_first=True,
            )

        if bidirectional:
            self.classifier = nn.Linear(hidden_dim * 2, 1)
        else:
            self.classifier = nn.Linear(hidden_dim, 1)
        self.dropout = nn.Dropout(dropout)

    def forward(self, inputs):
        embeddings = self.embedding(inputs)
        output, _ = self.model(embeddings)
        last_output = output[:, -1, :]
        last_output = self.dropout(last_output)
        logits = self.classifier(last_output)
        return logits

### 🔹 예제 6.30 사전 학습된 임베딩을 사용한 모델 학습
교재 결과: Val Accuracy 0.7974 (1에폭) → 0.8308 (5에폭)
- 참고: `from_pretrained`는 기본이 `freeze=True`라서 임베딩은 학습 중에 고정된다.

In [31]:
classifier = SentenceClassifier(
    n_vocab=n_vocab, hidden_dim=hidden_dim, embedding_dim=embedding_dim,
    n_layers=n_layers, pretrained_embedding=init_embeddings
).to(device)
criterion = nn.BCEWithLogitsLoss().to(device)
optimizer = optim.RMSprop(classifier.parameters(), lr=0.001)

epochs = 5
interval = 500

for epoch in range(epochs):
    train(classifier, train_loader, criterion, optimizer, device, interval)
    test(classifier, test_loader, criterion, device)

Train Loss 0 : 0.6933421492576599
Train Loss 500 : 0.6945525363176882
Train Loss 1000 : 0.6866570839455554
Train Loss 1500 : 0.6568564645534035
Train Loss 2000 : 0.6424674766621311
Train Loss 2500 : 0.6322608440876103
Val Loss : 0.5737272278664592, Val Accuracy : 0.7398
Train Loss 0 : 0.5969178080558777
Train Loss 500 : 0.5460646645335142
Train Loss 1000 : 0.5359399585159389
Train Loss 1500 : 0.5339997802115852
Train Loss 2000 : 0.5264097566398486
Train Loss 2500 : 0.5270275391194402
Val Loss : 0.46811389685057986, Val Accuracy : 0.7798
Train Loss 0 : 0.599204421043396
Train Loss 500 : 0.49585537704641
Train Loss 1000 : 0.4890807743642952
Train Loss 1500 : 0.4839970305770497
Train Loss 2000 : 0.4780050480309157
Train Loss 2500 : 0.47531157506198607
Val Loss : 0.452736658243516, Val Accuracy : 0.7912
Train Loss 0 : 0.5693593621253967
Train Loss 500 : 0.45928342809934103
Train Loss 1000 : 0.4499521888978593
Train Loss 1500 : 0.4442903385420071
Train Loss 2000 : 0.44393815828868
Train Los